In [ ]:
!pip install polars tqdm geopandas

In [ ]:
import pandas as pd
import numpy as np
import os
import tqdm
import pydantic
import polars as pl
from scipy.interpolate import interp1d
import geopandas as gpd
from sklearn.preprocessing import QuantileTransformer
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split

In [ ]:
combined_df = pd.read_csv("/content/drive/MyDrive/preprocessed/combined.csv")
combined_df

/tmp/ipykernel_15682/750020648.py:1: DtypeWarning: Columns (1,8,9,10,11,12,14,15,17,18,19,20,22,23,24,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,45,46,48,49,51,52,54,55,57,58,59,60,61,62,63,64,65,66,67,68,70,71,73,74,75,76,77,78,79,81,82,83,84,86,87,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,106,107,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,128,129,131,132,133,134,135,136,137,138,139,140,141,142,143,145,146,147,148,149,150,151,152,153,155,156,157,158,160,161,162,163,164,165,166,167,168,169,170,171,172,173) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_df = pd.read_csv("/content/drive/MyDrive/preprocessed/combined.csv")


,SID,SEASON,NUMBER,BASIN,SUBBASIN,NAME,ISO_TIME,NATURE,LAT,LON,...,STORM_SPEED,STORM_DIR,prev_lat,prev_lon,dist_km,time_diff_hr,speed_kmh,elevation,slope,aspect
0,1884186N16125,1884,16.0,WP,MM,UNNAMED,1884-07-03 16:00:00,TS,16.1,125.2,...,9,270,NaN,NaN,NaN,NaN,NaN,-2.147484e+09,0.0,0.0
1,1884186N16125,1884,16.0,WP,MM,UNNAMED,1884-07-03 18:00:00,TS,16.1,124.9,...,9,270,16.1,125.2,32.050127,2.0,16.025064,-2.147484e+09,0.0,0.0
2,1884186N16125,1884,16.0,WP,MM,UNNAMED,1884-07-03 21:00:00,TS,16.1,124.4,...,10,275,16.1,124.9,53.416871,3.0,17.805624,-2.147484e+09,0.0,0.0
3,1884186N16125,1884,16.0,WP,MM,UNNAMED,1884-07-04 00:00:00,TS,16.2,123.9,...,10,275,16.1,124.4,54.548739,3.0,18.182913,-2.147484e+09,0.0,0.0
4,1884186N16125,1884,16.0,WP,MM,UNNAMED,1884-07-04 03:00:00,TS,16.2,123.3,...,11,275,16.2,123.9,64.067849,3.0,21.355950,-2.147484e+09,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246662,2021148N05142,2021,33.0,WP,MM,CHOI-WAN,2021-06-05 12:00:00,MX,28.1,128.6,...,25,65,27.4,127.3,149.744966,3.0,49.914989,0.000000e+00,0.0,0.0
246663,2021148N05142,2021,33.0,WP,MM,CHOI-WAN,2021-06-05 15:00:00,ET,28.5,129.8,...,25,70,28.1,128.6,125.622145,3.0,41.874048,0.000000e+00,0.0,0.0
246664,2021148N05142,2021,33.0,WP,MM,CHOI-WAN,2021-06-05 18:00:00,ET,29.0,131.3,...,30,70,28.5,129.8,156.442146,3.0,52.147382,0.000000e+00,0.0,0.0
246665,2021148N05142,2021,33.0,WP,MM,CHOI-WAN,2021-06-05 21:00:00,ET,29.5,133.1,...,34,70,29.0,131.3,183.264921,3.0,61.088307,0.000000e+00,0.0,0.0


In [ ]:
merged_df = pd.read_csv("/content/drive/MyDrive/preprocessed/merged.csv")
merged_df

,1.000,2.000,3.000
0,1.0,2.0,3.0
1,1.0,2.0,3.0
2,1.0,2.0,3.0
3,1.0,2.0,3.0
4,1.0,2.0,3.0
...,...,...,...
62,1.0,2.0,3.0
63,1.0,2.0,3.0
64,1.0,2.0,3.0
65,1.0,2.0,3.0


In [ ]:
population_df = pd.read_csv("/content/drive/MyDrive/preprocessed/population_full_cleaned.csv")
population_df


,Name,Status,Population
0,Aklan,Province,634422
1,Altavas,Municipality,26181
2,Balete,Municipality,30310
3,Banga,Municipality,41188
4,Batan,Municipality,33932
...,...,...,...
746,San Mateo,Municipality,67433
747,San Pablo,Municipality,26462
748,Santiago,City,150313
749,Santo Tomas,Municipality,25997


In [ ]:
#Cleaning The Data
#Checking for columns
list(combined_df.columns)

['SID',
 'SEASON',
 'NUMBER',
 'BASIN',
 'SUBBASIN',
 'NAME',
 'ISO_TIME',
 'NATURE',
 'LAT',
 'LON',
 'WMO_WIND',
 'WMO_PRES',
 'WMO_AGENCY',
 'TRACK_TYPE',
 'DIST2LAND',
 'LANDFALL',
 'IFLAG',
 'USA_AGENCY',
 'USA_ATCF_ID',
 'USA_LAT',
 'USA_LON',
 'USA_RECORD',
 'USA_STATUS',
 'USA_WIND',
 'USA_PRES',
 'USA_SSHS',
 'USA_R34_NE',
 'USA_R34_SE',
 'USA_R34_SW',
 'USA_R34_NW',
 'USA_R50_NE',
 'USA_R50_SE',
 'USA_R50_SW',
 'USA_R50_NW',
 'USA_R64_NE',
 'USA_R64_SE',
 'USA_R64_SW',
 'USA_R64_NW',
 'USA_POCI',
 'USA_ROCI',
 'USA_RMW',
 'USA_EYE',
 'TOKYO_LAT',
 'TOKYO_LON',
 'TOKYO_GRADE',
 'TOKYO_WIND',
 'TOKYO_PRES',
 'TOKYO_R50_DIR',
 'TOKYO_R50_LONG',
 'TOKYO_R50_SHORT',
 'TOKYO_R30_DIR',
 'TOKYO_R30_LONG',
 'TOKYO_R30_SHORT',
 'TOKYO_LAND',
 'CMA_LAT',
 'CMA_LON',
 'CMA_CAT',
 'CMA_WIND',
 'CMA_PRES',
 'HKO_LAT',
 'HKO_LON',
 'HKO_CAT',
 'HKO_WIND',
 'HKO_PRES',
 'KMA_LAT',
 'KMA_LON',
 'KMA_CAT',
 'KMA_WIND',
 'KMA_PRES',
 'KMA_R50_DIR',
 'KMA_R50_LONG',
 'KMA_R50_SHORT',
 'KMA_R30_D

In [ ]:
print(merged_df.columns)
print(population_df.columns)

Index(['1.000', '2.000', '3.000'], dtype='object')
Index(['Name', 'Status', 'Population'], dtype='object')


In [ ]:
total_population = population_df["Population"].sum()
mean_population  = population_df["Population"].mean()

combined_df["total_population"] = total_population
combined_df["mean_population"]  = mean_population
combined_df["pop_density_proxy"] = mean_population / total_population

In [ ]:
risk_raw = -pd.to_numeric(combined_df["LANDFALL"], errors='coerce')
qt = QuantileTransformer(output_distribution="uniform")
risk_quantile = qt.fit_transform(risk_raw.values.reshape(-1, 1)).ravel()

# Create a temporary DataFrame to align risk_quantile and numeric LANDFALL, and drop NaNs
temp_df = pd.DataFrame({
    'risk_quantile_col': risk_quantile,
    'landfall_value_col': pd.to_numeric(combined_df["LANDFALL"], errors='coerce')
}).dropna()

# Extract the cleaned X and Y for splitting
X_for_split = temp_df['risk_quantile_col'].values
y_for_split = temp_df['landfall_value_col'].values

rq_train, rq_val, y_train, y_val = train_test_split(
    X_for_split, # This is now clean and numeric
    y_for_split, # This is now clean and numeric
    test_size=0.2,
    random_state=42
)


threshold = np.percentile(y_train, 10)

# soft labeling (y_train and y_val are already clean and numeric)
pseudo_event_train = 1 / (1 + np.exp((y_train - threshold)/10))
pseudo_event_val   = 1 / (1 + np.exp((y_val - threshold)/10))

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(rq_val, pseudo_event_val)


non_nan_risk_quantile_mask = ~np.isnan(risk_quantile)
risk_quantile_for_predict = risk_quantile[non_nan_risk_quantile_mask]

# Predict only on non-NaN values
event_prob_temp = iso.predict(risk_quantile_for_predict)

event_prob = np.full_like(risk_quantile, np.nan)
event_prob[non_nan_risk_quantile_mask] = event_prob_temp

combined_df["event_prob"] = event_prob

combined_df["event"] = (event_prob >= 0.5).astype(int)

combined_df["ISO_TIME"] = pd.to_datetime(combined_df["ISO_TIME"])

combined_df["duration"] = (
    combined_df.groupby("SID")["ISO_TIME"]
    .transform(lambda x: (x - x.min()).dt.total_seconds() / 3600)
)

In [ ]:
risk_df = combined_df.copy()

risk_df["risk_score"] = pd.to_numeric(combined_df["LANDFALL"], errors='coerce') #Ensure risk_score is numeric

In [ ]:
r = pd.to_numeric(risk_df["risk_score"], errors='coerce').values
y = combined_df["event"].values

n_bins = 10

r_for_bins = r[~np.isnan(r)]

# Ensure bins can be created even if r_for_bins is empty or has too few unique values
if len(r_for_bins) < 2:
    # If no valid data or only one unique value, create a minimal bin range
    min_val = r_for_bins.min() if len(r_for_bins) > 0 else 0.0
    max_val = r_for_bins.max() if len(r_for_bins) > 0 else 1.0
    if min_val == max_val:

        bins = np.array([min_val - 0.1, min_val + 0.1])
    else:
        bins = np.array([min_val, max_val])
else:
    bins = np.quantile(r_for_bins, np.linspace(0, 1, n_bins + 1))
    bins = np.unique(bins) # Ensure bins are strictly increasing

bin_ids = np.digitize(r, bins, right=True)

In [ ]:
bin_centers = []
bin_true = []
bin_brier = []
bin_counts = []

for b in range(1, len(bins)):
    mask = bin_ids == b
    if np.sum(mask) < 5:  # avoid unstable bins
        continue

    r_bin = r[mask]
    y_bin = y[mask]

    # bin prediction = mean risk score
    p_bin = r_bin.mean()

    # empirical event rate
    y_mean = y_bin.mean()

    # Brier error in this region
    brier = np.mean((r_bin - y_bin) ** 2)

    bin_centers.append(p_bin)
    bin_true.append(y_mean)
    bin_brier.append(brier)
    bin_counts.append(np.sum(mask))

weights = 1 / (np.array(bin_brier) + 1e-6)

weighted_true = np.array(bin_true) * weights
weighted_centers = np.array(bin_centers)

calibrator = interp1d(
    weighted_centers,
    weighted_true,
    kind="linear",
    bounds_error=False,
    fill_value="extrapolate"
)

risk_df["risk_score"] = calibrator(r)

In [ ]:
print("risk_score nunique:", len(np.unique(r)))
print("risk_score shape:", r.shape)
print("any NaN:", np.isnan(r).any())
print("event distribution:", np.bincount(y.astype(int)))

risk_score nunique: 3111
risk_score shape: (246667,)
any NaN: True
event distribution: [221206  25461]


In [ ]:
combined_df

,SID,SEASON,NUMBER,BASIN,SUBBASIN,NAME,ISO_TIME,NATURE,LAT,LON,...,speed_kmh,elevation,slope,aspect,total_population,mean_population,pop_density_proxy,event_prob,event,duration
0,1884186N16125,1884,16.0,WP,MM,UNNAMED,1884-07-03 16:00:00,TS,16.1,125.2,...,NaN,-2.147484e+09,0.0,0.0,105788416,140863.403462,0.001332,7.602187e-11,0,0.0
1,1884186N16125,1884,16.0,WP,MM,UNNAMED,1884-07-03 18:00:00,TS,16.1,124.9,...,16.025064,-2.147484e+09,0.0,0.0,105788416,140863.403462,0.001332,6.860984e-10,0,2.0
2,1884186N16125,1884,16.0,WP,MM,UNNAMED,1884-07-03 21:00:00,TS,16.1,124.4,...,17.805624,-2.147484e+09,0.0,0.0,105788416,140863.403462,0.001332,1.374508e-07,0,5.0
3,1884186N16125,1884,16.0,WP,MM,UNNAMED,1884-07-04 00:00:00,TS,16.2,123.9,...,18.182913,-2.147484e+09,0.0,0.0,105788416,140863.403462,0.001332,2.753569e-05,0,8.0
4,1884186N16125,1884,16.0,WP,MM,UNNAMED,1884-07-04 03:00:00,TS,16.2,123.3,...,21.355950,-2.147484e+09,0.0,0.0,105788416,140863.403462,0.001332,1.840719e-04,0,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246662,2021148N05142,2021,33.0,WP,MM,CHOI-WAN,2021-06-05 12:00:00,MX,28.1,128.6,...,49.914989,0.000000e+00,0.0,0.0,105788416,140863.403462,0.001332,4.193796e-13,0,204.0
246663,2021148N05142,2021,33.0,WP,MM,CHOI-WAN,2021-06-05 15:00:00,ET,28.5,129.8,...,41.874048,0.000000e+00,0.0,0.0,105788416,140863.403462,0.001332,1.530893e-10,0,207.0
246664,2021148N05142,2021,33.0,WP,MM,CHOI-WAN,2021-06-05 18:00:00,ET,29.0,131.3,...,52.147382,0.000000e+00,0.0,0.0,105788416,140863.403462,0.001332,2.789468e-10,0,210.0
246665,2021148N05142,2021,33.0,WP,MM,CHOI-WAN,2021-06-05 21:00:00,ET,29.5,133.1,...,61.088307,0.000000e+00,0.0,0.0,105788416,140863.403462,0.001332,6.896549e-12,0,213.0


In [ ]:
combined_df["month"] = pd.to_datetime(combined_df["ISO_TIME"]).dt.month
combined_df["seasonal_phase"] = combined_df["month"] // 3  # crude climate phase proxy

In [ ]:
combined_df["abs_lat"] = pd.to_numeric(combined_df["LAT"], errors='coerce').abs()
combined_df["lat_band"] = pd.cut(
    combined_df["abs_lat"],
    bins=[0, 10, 20, 30, 90],
    labels=[1,2,3,4]
)

In [ ]:
combined_df["monsoon_proxy"] = (
    pd.to_numeric(combined_df["STORM_SPEED"], errors='coerce') *
    np.sin(np.radians(pd.to_numeric(combined_df["STORM_DIR"], errors='coerce')))
)

In [ ]:
combined_df["delta_lat"] = pd.to_numeric(combined_df["LAT"], errors='coerce') - pd.to_numeric(combined_df["prev_lat"], errors='coerce')
combined_df["delta_lon"] = pd.to_numeric(combined_df["LON"], errors='coerce') - pd.to_numeric(combined_df["prev_lon"], errors='coerce')

combined_df["advection_intensity"] = np.sqrt(
    combined_df["delta_lat"]**2 + combined_df["delta_lon"]**2
)

In [ ]:
combined_df["aerosol_proxy"] = (
    pd.to_numeric(combined_df["WMO_WIND"], errors='coerce') *
    (1 / (1 + pd.to_numeric(combined_df["DIST2LAND"], errors='coerce')))
)

In [ ]:
combined_df["risk_score"] = risk_df["risk_score"]
combined_df["event_prob"] = risk_df["event_prob"]
model_df = combined_df[
    [
        #Time
        "SID", "ISO_TIME",
        # physics
        "WMO_WIND", "WMO_PRES",
        "STORM_SPEED", "STORM_DIR",

        # position
        "LAT", "LON", "DIST2LAND",
        "abs_lat", "lat_band",

        # temporal dynamics
        "time_diff_hr", "month", "seasonal_phase",
        "prev_lat", "prev_lon",
        "delta_lat", "delta_lon",
        "advection_intensity",

        # climate proxies
        "monsoon_proxy",
        "aerosol_proxy",

        # terrain
        "elevation", "slope", "aspect",

        # exposure
        "risk_score", "total_population",

        # target
        "event", "event_prob", "duration"
    ]
].copy()

In [ ]:
model_df

,SID,ISO_TIME,WMO_WIND,WMO_PRES,STORM_SPEED,STORM_DIR,LAT,LON,DIST2LAND,abs_lat,...,monsoon_proxy,aerosol_proxy,elevation,slope,aspect,risk_score,total_population,event,event_prob,duration
0,1884186N16125,1884-07-03 16:00:00,NaN,NaN,9,270,16.1,125.2,254,16.1,...,-9.000000,NaN,-2.147484e+09,0.0,0.0,0.0,105788416,0,7.602187e-11,0.0
1,1884186N16125,1884-07-03 18:00:00,NaN,NaN,9,270,16.1,124.9,243,16.1,...,-9.000000,NaN,-2.147484e+09,0.0,0.0,0.0,105788416,0,6.860984e-10,2.0
2,1884186N16125,1884-07-03 21:00:00,NaN,NaN,10,275,16.1,124.4,221,16.1,...,-9.961947,NaN,-2.147484e+09,0.0,0.0,0.0,105788416,0,1.374508e-07,5.0
3,1884186N16125,1884-07-04 00:00:00,NaN,NaN,10,275,16.2,123.9,168,16.2,...,-9.961947,NaN,-2.147484e+09,0.0,0.0,0.0,105788416,0,2.753569e-05,8.0
4,1884186N16125,1884-07-04 03:00:00,NaN,NaN,11,275,16.2,123.3,115,16.2,...,-10.958142,NaN,-2.147484e+09,0.0,0.0,0.0,105788416,0,1.840719e-04,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246662,2021148N05142,2021-06-05 12:00:00,NaN,1002.0,25,65,28.1,128.6,378,28.1,...,22.657695,NaN,0.000000e+00,0.0,0.0,0.0,105788416,0,4.193796e-13,204.0
246663,2021148N05142,2021-06-05 15:00:00,NaN,NaN,25,70,28.5,129.8,292,28.5,...,23.492316,NaN,0.000000e+00,0.0,0.0,0.0,105788416,0,1.530893e-10,207.0
246664,2021148N05142,2021-06-05 18:00:00,NaN,1004.0,30,70,29.0,131.3,230,29.0,...,28.190779,NaN,0.000000e+00,0.0,0.0,0.0,105788416,0,2.789468e-10,210.0
246665,2021148N05142,2021-06-05 21:00:00,NaN,NaN,34,70,29.5,133.1,267,29.5,...,31.949549,NaN,0.000000e+00,0.0,0.0,0.0,105788416,0,6.896549e-12,213.0


##Proper Feature Setting and Tunning

In [ ]:
model_df["ISO_TIME"] = pd.to_datetime(model_df["ISO_TIME"])
# Ensure WMO_WIND is numeric for aggregation
model_df["WMO_WIND"] = pd.to_numeric(model_df["WMO_WIND"], errors='coerce')

storm_df = (
    model_df.groupby("SID")
    .agg({
        "ISO_TIME": "min",   # storm start time
        "event": "max",
        "duration": "max",
        "WMO_WIND": "max"
    })
    .reset_index()
)

storm_df = storm_df.sort_values("ISO_TIME").reset_index(drop=True)

In [ ]:
n = len(storm_df)

train_end = int(n * 0.7)
val_end   = int(n * 0.85)

In [ ]:
train_sids = storm_df.iloc[:train_end]["SID"].values
val_sids   = storm_df.iloc[train_end:val_end]["SID"].values
test_sids  = storm_df.iloc[val_end:]["SID"].values

In [ ]:
train_df = model_df[model_df["SID"].isin(train_sids)].copy()
val_df   = model_df[model_df["SID"].isin(val_sids)].copy()
test_df  = model_df[model_df["SID"].isin(test_sids)].copy()

##Sanity Checks

In [ ]:
combined_df["event"].value_counts()

,count
event,
0,221206
1,25461


#Another Set for Ablation and Future Experiments

In [ ]:
ablation_features = [
    "WMO_WIND", "WMO_PRES",
    "STORM_SPEED", "STORM_DIR",
    "LAT", "LON",
    "DIST2LAND",
    "time_diff_hr"
]

In [ ]:
abl_df_train = train_df[ablation_features + ["event", "event_prob", "duration"]].copy()
abl_df_val   = val_df[ablation_features + ["event", "event_prob", "duration"]].copy()
abl_df_test  = test_df[ablation_features + ["event", "event_prob", "duration"]].copy()

#Sanity Checks (to ensure no leakage)

In [ ]:
print(set(train_df["SID"]) & set(val_df["SID"]))
print(set(train_df["SID"]) & set(test_df["SID"]))
print(set(val_df["SID"]) & set(test_df["SID"]))

set()
set()
set()


In [ ]:
print(train_df["ISO_TIME"].min())
print(train_df["ISO_TIME"].max())
print(val_df["ISO_TIME"].min())
print(val_df["ISO_TIME"].max())
print(test_df["ISO_TIME"].min())
print(test_df["ISO_TIME"].max())

1884-06-24 16:00:00
1987-08-04 00:00:00
1987-07-21 00:00:00
2004-09-19 06:00:00
2004-09-18 12:00:00
2026-02-06 12:00:00


#Dataset Download and Upload to drive

In [ ]:
os.makedirs("/content/drive/MyDrive/preprocessed/official_data", exist_ok=True)

train_df.to_csv("/content/drive/MyDrive/preprocessed/official_data/train.csv", index=False)
val_df.to_csv("/content/drive/MyDrive/preprocessed/official_data/val.csv", index=False)
test_df.to_csv("/content/drive/MyDrive/preprocessed/official_data/test.csv", index=False)

In [ ]:
#For Ablation
os.makedirs("/content/drive/MyDrive/preprocessed/ablation", exist_ok=True)

abl_df_train.to_csv("/content/drive/MyDrive/preprocessed/ablation/abl_train.csv", index=False)
abl_df_val.to_csv("/content/drive/MyDrive/preprocessed/ablation/abl_val.csv", index=False)
abl_df_test.to_csv("/content/drive/MyDrive/preprocessed/ablation/abl_test.csv", index=False)